# Mark - Person Names Frequency List

This notebook uses the `grc_odycy_joint_trf` model to count the frequency of proper nouns in the Gospel of Mark.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="thinc.shims.pytorch")

import spacy
import torch
import collections

# Load the transformer model
nlp = spacy.load("grc_odycy_joint_trf")

# Read the Mark Greek text
with open("data/NT/Greek/Mark-greek.txt", "r", encoding="utf-8") as f:
    text = f.read()

import re
# Remove verse numbers to fix tokenization
text = re.sub(r'\d+', ' ', text)

doc = nlp(text)

propn_freq = collections.Counter()

for t in doc:
    # odycy model often tags Proper Nouns as NOUN, so we also check for Title Case
    if t.pos_ in ["PROPN", "NOUN"] and t.text and t.text[0].isupper():
        propn_freq[t.lemma_] += 1
    elif t.pos_ == "PROPN":
        propn_freq[t.lemma_] += 1

sorted_freq = propn_freq.most_common()

import unicodedata

def remove_accents(text):
    text = text.replace("(", "").replace(")", "").replace("2", "")
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn').lower()

lexicon_path = "STEPBible-Data/Lexicons/TBESG - Translators Brief lexicon of Extended Strongs for Greek - STEPBible.org CC BY.txt"
greek_to_english = {}
with open(lexicon_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("G") and "\t" in line:
            parts = line.split("\t")
            if len(parts) >= 7:
                greek_words = parts[3].strip().split(",")
                english_gloss = parts[6].strip()
                for gw in greek_words:
                    clean_gw = remove_accents(gw.strip())
                    if clean_gw not in greek_to_english:
                        greek_to_english[clean_gw] = english_gloss

greek_to_english["βηθσαιδαν"] = "Bethsaida"
greek_to_english["ιεροσολυμων"] = "Jerusalem"

print("Person Names (Proper Nouns) Frequency in Mark:")
for name, freq in sorted_freq:
    norm_name = remove_accents(name)
    english_name = greek_to_english.get(norm_name, "Unknown")
    print(f"{name} ({english_name}): {freq}")


Person Names (Proper Nouns) Frequency in Mark:
ἰησοῦς (Jesus): 82
ἰωάν(ν)ης (John): 26
πέτρος (Peter): 20
ἰάκωβος (James): 15
γαλιλαία (Galilee): 12
φαρισαῖος (Pharisee): 12
σίμων (Simon): 11
ἱεροσόλυμα (Jerusalem): 10
διδάσκαλος (teacher): 9
Πιλᾶτος (Pilate): 9
μωϋσῆς (Moses): 8
μαρία (Mary): 8
Ἠλίας (Elijah): 8
χριστός (Christ): 7
σατανᾶς (Satan): 6
ἡρῴδης (Herod): 6
ἰουδαῖος (Jew): 6
ἰορδάνης (Jordan): 4
ἀνδρέας (Andrew): 4
ζεβεδαῖος (Zebedee): 4
ναζαρηνός (Nazareth): 4
ἰούδας (Judas): 4
βηθανία (Bethany): 4
καῖσαρ (Caesar): 4
μαγδαληνή (Magdalene): 4
καφαρναούμ (Unknown): 3
Δαυίδ (David): 3
ἰουδαία (Judea): 3
τύρος (Tyre): 3
υἱός (son): 3
φίλιππος (Philip): 3
ἰωσῆς (Joseph): 3
Ῥαββίς (Unknown): 3
ὄρος (mountain): 3
ἐλαία (olive tree): 3
βαραββᾶς (Barabbas): 3
τέκνον (child): 2
ἁλφαῖος (Alphaeus): 2
ἡρῳδιανοί (Herodian): 2
σιδών (Sidon): 2
Ἡρῳδιάς (Herodias): 2
Ἡρῴδης (Herod): 2
βηθσαϊδά(ν) (Bethsaida): 2
ἰσραήλ (Israel): 2
Δαυὶδ (David): 2
σαλώμη (Salome): 2
ἰωσήφ (Joseph): 2
ἀρχή 